# Supervised Fine-Tuning (SFT) with PEFT of Amazon Nova 2.0 Lite using Amazon SageMaker Training Job

## Lab 3: Model deployment and inference

After training our model, we want to make it available for inference. Amazon Bedrock provides a serverless endpoint for model deployment, allowing us to serve the model without managing infrastructure.

The Bedrock Custom Model feature lets us import our fine-tuned model and access it through the same API as other foundation models.

### Prerequisites

In [ ]:
import sagemaker
import boto3

sess = sagemaker.Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

sagemaker_client = boto3.client("sagemaker")
bedrock = boto3.client("bedrock", region_name=sess.boto_region_name)

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
guardrail_id = None
model_arn = None
custom_model_arn = None

### Utility functions

Retrieve last completed job from SageMaker AI

In [ ]:
def get_last_job_name(job_name_prefix):
    matching_jobs = []
    next_token = None

    while True:
        search_params = {
            'Resource': 'TrainingJob',
            'SearchExpression': {
                'Filters': [
                    {
                        'Name': 'TrainingJobName',
                        'Operator': 'Contains',
                        'Value': job_name_prefix
                    },
                    {
                        'Name': 'TrainingJobStatus',
                        'Operator': 'Equals',
                        'Value': "Completed"
                    }
                ]
            },
            'SortBy': 'CreationTime',
            'SortOrder': 'Descending',
            'MaxResults': 100
        }

        if next_token:
            search_params['NextToken'] = next_token

        search_response = sagemaker_client.search(**search_params)

        matching_jobs.extend([
            job['TrainingJob']['TrainingJobName']
            for job in search_response['Results']
            if job['TrainingJob']['TrainingJobName'].startswith(job_name_prefix)
        ])

        next_token = search_response.get('NextToken')
        if not next_token or matching_jobs:
            break

    if not matching_jobs:
        print(f"No completed training jobs found starting with prefix '{job_name_prefix}'")
        return None

    return matching_jobs[0]

Get checkpoint configurations

In [ ]:
def get_sagemaker_checkpoint_s3_uri(training_job_name):
    sagemaker_cl = boto3.client('sagemaker')
    try:
        response = sagemaker_cl.describe_training_job(TrainingJobName=training_job_name)
        checkpoint_config = response['CheckpointConfig']['S3Uri']
        return checkpoint_config
    except Exception as e:
        print(f"Error retrieving checkpoint configuration: {e}")
        return None

Extract the right checkpoint path from the logs

In [ ]:
import os
import re

def filter_s3_paths(message, base_path):
    pattern = r's3://[^\s]+'
    matches = re.findall(pattern, message)
    return [match.rstrip('.') for match in matches if base_path in match]

def get_logs_containing_text(log_group_name, search_text, region='us-east-1'):
    logs_client = boto3.client('logs', region_name=region)
    matching_events = []

    paginator = logs_client.get_paginator('filter_log_events')
    for page in paginator.paginate(logGroupName=log_group_name):
        for event in page['events']:
            if search_text in event['message']:
                matching_events.append(event)

    return matching_events

Utility functions to check the manifest.json file

In [ ]:
import json
import os
import tarfile

def extract_tar_gz(tar_path, extract_to='.'):
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(extract_to)

def download_s3_file(s3_path, local_path):
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    s3 = boto3.client('s3')
    bucket = s3_path.split('/')[2]
    key = '/'.join(s3_path.split('/')[3:])
    s3.download_file(bucket, key, local_path)

def get_checkpoint_path(manifest_path):
    with open(manifest_path) as f:
        return json.load(f)['checkpoint_s3_bucket']

### Update model configurations

In [ ]:
base_job_name = "train-nova-2-lite-sft-peft"

job_name = get_last_job_name(base_job_name)

print("Job name:", job_name)

if job_name:
    model_path = get_sagemaker_checkpoint_s3_uri(job_name)

    print("Model path:", model_path)

    imported_model_name = "nova-lite-2-sagemaker-sft-peft-reasoning"

### Creating the Bedrock Custom Model

In [ ]:
from botocore.exceptions import ClientError

try:
    request_params = {
        "modelName": imported_model_name,
        "modelSourceConfig": {
            "s3DataSource": {
                "s3Uri": model_path,
            }
        },
        "roleArn": role,
        "clientRequestToken": "NovaLite2RecipeSageMaker",
    }
    response = bedrock.create_custom_model(**request_params)
    model_arn = response["modelArn"]

    print("/***************************************************/")
    print(f"Model import job created with ARN: {model_arn}")
    print("/***************************************************/")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException':
        print("S3 URI invalid, downloading manifest to find correct path...")

        try:
            download_s3_file(
                f"s3://{sess.default_bucket()}/{base_job_name}/{job_name}/output/output.tar.gz",
                "./tmp_output/output.tar.gz",
            )
            extract_tar_gz('./tmp_output/output.tar.gz', './tmp_output')
            model_path = get_checkpoint_path('./tmp_output/manifest.json')

            request_params["modelSourceConfig"]["s3DataSource"]["s3Uri"] = model_path
            response = bedrock.create_custom_model(**request_params)
            model_arn = response["modelArn"]

            print("/***************************************************/")
            print(f"Model import job created with ARN: {model_arn}")
            print("/***************************************************/")
        except:
            print("Manifest not found, searching CloudWatch logs for correct path...")

            logs = get_logs_containing_text(
                '/aws/sagemaker/TrainingJobs',
                model_path
            )

            for log in logs:
                s3_paths = filter_s3_paths(log['message'], model_path)
                if s3_paths:
                    model_path = s3_paths[0]
                    print(f"Found S3 path in logs: {model_path}")
                    break

            request_params["modelSourceConfig"]["s3DataSource"]["s3Uri"] = model_path
            response = bedrock.create_custom_model(**request_params)
            model_arn = response["modelArn"]

            print("/***************************************************/")
            print(f"Model import job created with ARN: {model_arn}")
            print("/***************************************************/")
    else:
        raise

### Monitoring the Model status

After initiating the model import, we need to monitor its progress. The status goes through several states:

* CREATING: Model is being imported
* ACTIVE: Import successful
* FAILED: Import encountered errors

In [ ]:
from IPython.display import clear_output
import time

while True:
    response = bedrock.list_custom_models(sortBy='CreationTime', sortOrder='Descending')
    model_summaries = response["modelSummaries"]
    status = ""
    for model in model_summaries:
        if model["modelName"] == imported_model_name:
            status = model["modelStatus"].upper()
            model_arn = model["modelArn"]
            print(f'{model["modelStatus"].upper()} {model["modelArn"]} ...')
            if status in ["ACTIVE", "FAILED"]:
                break
    if status in ["ACTIVE", "FAILED"]:
        break
    clear_output(wait=True)
    time.sleep(10)

model_arn

### Deploy a custom model for on-demand inference

Please refer to the official [AWS Documentation](https://docs.aws.amazon.com/nova/latest/userguide/deploy-custom-model.html)

In [ ]:
request_params = {
    "clientRequestToken": "NovaLite2RecipeSageMakerODI",
    "modelDeploymentName": f"{imported_model_name}-odi",
    "modelArn": model_arn,
}

response = bedrock.create_custom_model_deployment(**request_params)

response

In [ ]:
from IPython.display import clear_output
import time

while True:
    response = bedrock.list_custom_model_deployments(
        sortBy="CreationTime", sortOrder="Descending"
    )
    model_summaries = response["modelDeploymentSummaries"]
    status = ""
    for model in model_summaries:
        if model["customModelDeploymentName"] == f"{imported_model_name}-odi":
            status = model["status"].upper()
            custom_model_arn = model["customModelDeploymentArn"]
            print(f'{model["status"].upper()} {model["customModelDeploymentArn"]} ...')
            if status in ["CREATING"]:
                break
    if status in ["ACTIVE", "FAILED"]:
        break
    clear_output(wait=True)
    time.sleep(10)

custom_model_arn

***

### Testing the Deployed Model

Now that our model is deployed to Amazon Bedrock, we can invoke it for inference.

Amazon Nova 2 Lite supports [extended thinking](https://docs.aws.amazon.com/nova/latest/nova2-userguide/reasoning-capabilities.html), which is disabled by default. When enabled via `reasoningConfig`, the model generates internal reasoning tokens that improve response quality. Note that reasoning content is currently redacted in the output (`[REDACTED]`), though you are still charged for these tokens.

In [ ]:
from botocore.config import Config

client = boto3.client(
    service_name="bedrock-runtime",
    region_name=sess.boto_region_name,
    config=Config(
        connect_timeout=300,
        read_timeout=300,
        retries={"max_attempts": 3},
    ),
)

In [ ]:
import time


def generate(
    model_id,
    messages,
    system_prompt=None,
    tools=None,
    temperature=0.3,
    max_tokens=4096,
    top_p=0.9,
    guardrail_id=None,
    reasoning_config=None,
    max_retries=10,
):
    kwargs = {}

    if reasoning_config:
        kwargs["additionalModelRequestFields"] = {
            "reasoningConfig": reasoning_config
        }
    else:
        kwargs["inferenceConfig"] = {
            "temperature": temperature,
            "maxTokens": max_tokens,
            "topP": top_p,
        }

    if tools:
        kwargs["toolConfig"] = tools
    if system_prompt:
        kwargs["system"] = [{"text": system_prompt}]
    if guardrail_id:
        kwargs["guardrailConfig"] = {
            "guardrailIdentifier": guardrail_id,
            "guardrailVersion": "1",
        }

    for attempt in range(max_retries):
        try:
            return client.converse(modelId=model_id, messages=messages, **kwargs)
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {str(e)}")
            if attempt < max_retries - 1:
                time.sleep(30)
            else:
                print("Max retries reached. Unable to get response.")
                return None

Use the custom model deployment ARN created with the Bedrock Custom Model

In [ ]:
import json
import textwrap

model_arn = (
    custom_model_arn
    if custom_model_arn is not None
    else "<BEDROCK_CUSTOM_MODEL_DEPLOYMENT_ARN>"
)

system_prompt = """
You are an AI assistant that reasons in {language} and responds in English.

Always reason through the problem in {language}, then provide your final answer in English.
"""

system_prompt = system_prompt.format(language="Spanish")

messages = [
    {
        "role": "user",
        "content": [
            {
                "text": "Hello, how are you?"
            }
        ],
    },
]

response = generate(
    model_id=model_arn,
    system_prompt=textwrap.dedent(system_prompt).strip(),
    messages=messages,
    guardrail_id=guardrail_id,
    reasoning_config={"type": "enabled", "maxReasoningEffort": "high"},
)

for block in response["output"]["message"]["content"]:
    if "reasoningContent" in block:
        print("[Reasoning]")
        print(block["reasoningContent"]["reasoningText"]["text"])
        print()
    elif "text" in block:
        print("[Response]")
        print(block["text"])

print(f"\nToken usage: {response['usage']}")